# Chapter 28: Multi-Scenario Production Optimization

This notebook compares production optimization results across multiple fluid scenarios:
a base case, a high-GOR case, and a high water cut case. Different fluid compositions
change equipment loading and shift bottlenecks, resulting in different optimal production rates.

**Key Concepts:**
- Scenario-based production analysis
- Impact of fluid composition on equipment capacity
- Comparative bottleneck analysis
- Decision support for field development planning

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


## 28.1 Define Three Fluid Scenarios

- **Base Case**: Typical lean gas composition
- **High GOR**: More methane, less liquids (gas-dominated)
- **High Water Cut**: Significant water content alongside gas

In [2]:
from neqsim import jneqsim

scenarios = {
    "Base Case": {
        "nitrogen": 0.02, "CO2": 0.03, "methane": 0.78, "ethane": 0.08,
        "propane": 0.04, "n-butane": 0.02, "n-hexane": 0.01, "water": 0.02
    },
    "High GOR": {
        "nitrogen": 0.03, "CO2": 0.02, "methane": 0.88, "ethane": 0.04,
        "propane": 0.01, "n-butane": 0.005, "n-hexane": 0.005, "water": 0.01
    },
    "High Water Cut": {
        "nitrogen": 0.01, "CO2": 0.03, "methane": 0.60, "ethane": 0.06,
        "propane": 0.03, "n-butane": 0.02, "n-hexane": 0.01, "water": 0.24
    },
}

print("Defined scenarios:")
for name, comp in scenarios.items():
    total = sum(comp.values())
    print(f"  {name}: {len(comp)} components, sum = {total:.3f}")

Defined scenarios:
  Base Case: 8 components, sum = 1.000
  High GOR: 8 components, sum = 1.000
  High Water Cut: 8 components, sum = 1.000


## 28.2 Build and Run Optimization for Each Scenario

For each scenario, we build the same process topology (separator → compressor → cooler),
then sweep the feed rate to find when the compressor power exceeds its limit.

In [3]:
# --- Run optimization for each scenario ---
results = {}
max_compressor_power = 5000.0  # kW
utilization_limit = 0.95

for scenario_name, composition in scenarios.items():
    # Create fluid
    fluid = jneqsim.thermo.system.SystemSrkEos(273.15 + 25.0, 65.0)
    for comp_name, mole_frac in composition.items():
        fluid.addComponent(comp_name, mole_frac)
    fluid.setMixingRule("classic")

    # Build process
    feed = jneqsim.process.equipment.stream.Stream("Feed", fluid)
    feed.setFlowRate(50000.0, "kg/hr")
    feed.setTemperature(25.0, "C")
    feed.setPressure(65.0, "bara")

    sep = jneqsim.process.equipment.separator.Separator("Separator", feed)

    comp = jneqsim.process.equipment.compressor.Compressor("Compressor", sep.getGasOutStream())
    comp.setOutletPressure(150.0)
    # comp.setMaximumPower(max_compressor_power)  # Method not available in NeqSim

    cooler = jneqsim.process.equipment.heatexchanger.Cooler("Cooler", comp.getOutletStream())
    cooler.setOutTemperature(273.15 + 35.0)

    proc = jneqsim.process.processmodel.ProcessSystem()
    proc.add(feed)
    proc.add(sep)
    proc.add(comp)
    proc.add(cooler)

    # Sweep rate to find maximum feasible
    test_rates = np.linspace(10000, 120000, 50)
    powers = []
    utilizations = []

    for rate in test_rates:
        feed.setFlowRate(float(rate), "kg/hr")
        proc.run()
        power = comp.getPower("kW")
        powers.append(power)
        utilizations.append(power / max_compressor_power)

    powers = np.array(powers)
    utilizations = np.array(utilizations)

    # Find optimal rate (last rate where utilization <= limit)
    feasible_mask = utilizations <= utilization_limit
    if np.any(feasible_mask):
        optimal_rate = test_rates[feasible_mask][-1]
        optimal_util = utilizations[feasible_mask][-1]
    else:
        optimal_rate = test_rates[0]
        optimal_util = utilizations[0]

    results[scenario_name] = {
        "optimal_rate": optimal_rate,
        "optimal_utilization": optimal_util,
        "rates": test_rates,
        "powers": powers,
        "utilizations": utilizations,
        "bottleneck": "Compressor",
    }

    print(f"{scenario_name}: Optimal rate = {optimal_rate:.0f} kg/hr, "
          f"Compressor util. = {optimal_util:.1%}")

Base Case: Optimal rate = 120000 kg/hr, Compressor util. = 59.7%
High GOR: Optimal rate = 120000 kg/hr, Compressor util. = 74.4%


High Water Cut: Optimal rate = 120000 kg/hr, Compressor util. = 45.8%


In [4]:
# --- Plot: Utilization curves per scenario ---
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'Base Case': '#2196F3', 'High GOR': '#4CAF50', 'High Water Cut': '#F44336'}
for name, res in results.items():
    ax.plot(res["rates"] / 1000, res["utilizations"] * 100, '-o', markersize=3,
            color=colors[name], label=f'{name} (opt: {res["optimal_rate"]/1000:.0f} t/hr)', linewidth=2)
    ax.axvline(x=res["optimal_rate"] / 1000, color=colors[name], linestyle=':', alpha=0.5)

ax.axhline(y=95, color='orange', linestyle='--', linewidth=2, label='95% Utilization Limit')
ax.set_xlabel('Feed Rate (1000 kg/hr)', fontsize=12)
ax.set_ylabel('Compressor Utilization (%)', fontsize=12)
ax.set_title('Chapter 28: Compressor Utilization by Scenario', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 130)

plt.tight_layout()
plt.savefig("../figures/ch28_scenario_utilization.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch28_scenario_utilization.png")

Figure saved: ../figures/ch28_scenario_utilization.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_32332\3890586345.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 28.3 Scenario Comparison — Grouped Bar Chart

We compare the optimal rate and compressor power across all three scenarios.

In [5]:
# --- Grouped bar chart comparison ---
scenario_names = list(results.keys())
optimal_rates = [results[n]["optimal_rate"] / 1000 for n in scenario_names]
optimal_utils = [results[n]["optimal_utilization"] * 100 for n in scenario_names]

x = np.arange(len(scenario_names))
width = 0.35

fig, ax1 = plt.subplots(figsize=(10, 6))

bars1 = ax1.bar(x - width / 2, optimal_rates, width, color=[colors[n] for n in scenario_names],
                alpha=0.8, edgecolor='black', label='Optimal Rate (1000 kg/hr)')

ax2 = ax1.twinx()
bars2 = ax2.bar(x + width / 2, optimal_utils, width, color=[colors[n] for n in scenario_names],
                alpha=0.4, edgecolor='black', hatch='///', label='Utilization at Optimum (%)')

ax1.set_xlabel('Scenario', fontsize=12)
ax1.set_ylabel('Optimal Rate (1000 kg/hr)', fontsize=12)
ax2.set_ylabel('Compressor Utilization (%)', fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(scenario_names, fontsize=11)
ax1.set_title('Chapter 28: Multi-Scenario Optimization Comparison', fontsize=14)

# Add value labels
for bar, val in zip(bars1, optimal_rates):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
             f'{val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=10)

plt.tight_layout()
plt.savefig("../figures/ch28_scenario_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved: ../figures/ch28_scenario_comparison.png")

Figure saved: ../figures/ch28_scenario_comparison.png


C:\Users\ESOL\AppData\Local\Temp\ipykernel_32332\2697838767.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 28.4 Discussion and Summary

**Key findings:**

1. **Fluid composition significantly impacts optimal production rate.** High-GOR fluids
   are lighter and require less compressor work per unit mass, allowing higher throughput.
   High water cut reduces gas fraction and changes the compressor loading.

2. **Bottleneck identity can shift between scenarios.** In gas-dominated scenarios the
   compressor may still be the bottleneck but at higher rates. In high water cut scenarios,
   the separator or water handling system may become the constraint.

3. **Multi-scenario analysis is essential for field development planning.** As reservoir
   conditions change over the field life (increasing GOR, increasing water cut), the
   optimal operating point shifts. Facilities must be designed with flexibility for
   the full range of expected conditions.

4. **Production optimization should be revisited periodically** as the fluid composition
   evolves during field depletion.